# Multi-gNB scenario: two RFsim gNBs sharing one Open5GS core

This example combines the working monolithic and CU/DU examples:

- `gnb`: monolithic OAI gNB with ten RFsim UEs.
- `cu` + `du`: a second OAI gNB with ten RFsim UEs.
- One shared Open5GS core, subscriber database and monitoring stack.

The two radio links use distinct simulated RUs, gNB identities and cell identities.
Every UE has its own IMSI and is assigned to exactly one radio endpoint.
The split gNB's N3 probe is on the CU; the monolithic probe is on `gnb`.

Run cells in order. Provisioning creates real resources. The default layout has
**eight VMs**: core/monitoring, gNB, CU, DU and four UE workers (five UEs each).
Adjust worker count and VM flavor below for your available capacity. Twenty active
RFsim UEs are a new scale test; this example has only been checked offline.

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git@main"
# CSI support is in this repository (branch csi): install it in editable mode.
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r "{REPOSITORY / 'ansible' / 'requirements.yaml'}"


Edit the project, infrastructure, VM flavor and UE worker count here.
The experiment name includes a timestamp so this example does not reuse another run's experiment.
Keep the SLICES CLI authenticated in this environment.

In [ ]:
from datetime import UTC, datetime
import asyncio
import json

PROJECT = "post5g-beta"
EXPERIMENT = "multi-gnb-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
INFRA = "fr-sophia2-bi-vm1"
VM_FLAVOR = "large"
UES_PER_GNB = 10
UE_WORKERS = 4  # At least one; increase if UE workers run out of CPU or memory.

UE_GROUPS = {
    "monolithic": [f"mono_ue{i:02d}" for i in range(1, UES_PER_GNB + 1)],
    "split": [f"split_ue{i:02d}" for i in range(1, UES_PER_GNB + 1)],
}
CLIENTS = UE_GROUPS["monolithic"] + UE_GROUPS["split"]
UE_NODES = [f"node{i}" for i in range(5, 5 + UE_WORKERS)]
print(f"Experiment: {EXPERIMENT}; {4 + UE_WORKERS} VMs; {len(CLIENTS)} UEs")


The metamodel describes the network. `gnb` has gNB ID 1; `cu`/`du` share gNB ID 2.
A UE connects to the monolithic `gnb` or the split `du`. Both gNBs connect to the same AMF.
All twenty subscribers are defined before the database is populated.

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.model_validate(
    {
        "ran": {
            "components": {
                "gnb": {
                    "type": "gnb",
                    "config": {"gnbId": 1, "nrCellId": 1, "physicalCellId": 0, "ru": "mono_ru"},
                },
                "cu": {"type": "cu", "config": {"gnbId": 2, "nrCellId": 2}},
                "du": {
                    "type": "du",
                    "config": {
                        "gnbId": 2,
                        "gnbDuId": 2,
                        "nrCellId": 2,
                        "physicalCellId": 1,
                        "cu_cp": "cu",
                        "cu_up": "cu",
                        "ru": "split_ru",
                    },
                },
            },
        },
        "rus": {"mono_ru": {"type": "simulated"}, "split_ru": {"type": "simulated"}},
        "ues": {
            name: {
                "config": {
                    "supi": f"00101{100 + index:010d}",
                    "gnbs": ["gnb" if name in UE_GROUPS["monolithic"] else "du"],
                    "ru": "mono_ru" if name in UE_GROUPS["monolithic"] else "split_ru",
                }
            }
            for index, name in enumerate(CLIENTS)
        },
    }
)


The deployment model chooses OAI/Open5GS and places workloads explicitly.
`node1` runs the shared core, monitoring and iperf server; `node2` runs `gnb`,
`node3` runs `cu`, and `node4` runs `du`. UEs are distributed across the UE workers.

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.model_validate(
    {
        "params": {
            "provisioning": {
                "project": PROJECT,
                "experiment": EXPERIMENT,
                "infra": INFRA,
                "flavor": VM_FLAVOR,
                "extra_path": "~/slices/bin",
                "create_experiment": True,
            },
            "k8s": {
                "master_node": "node1",
                "local_kubeconfig_path": str(Path(f".k8s-{EXPERIMENT}.conf").resolve()),
            },
        },
        "nodes": {
            "node1": {"roles": ["core", "database"]},
            **{name: {"roles": ["ran"]} for name in ("node2", "node3", "node4")},
            **{name: {"roles": ["ue"]} for name in UE_NODES},
        },
        "ran": {
            "implementation": "oai",
            "components": {
                "gnb": {"type": "gnb", "node_name": "node2"},
                "cu": {"type": "cu", "node_name": "node3"},
                "du": {"type": "du", "node_name": "node4"},
            },
        },
        "ues": {
            name: {"type": "containerized", "implementation": "oai", "node_name": UE_NODES[index % len(UE_NODES)]}
            for index, name in enumerate(CLIENTS)
        },
    }
)


Create the VMs, then deploy Kubernetes. This is the first cell that provisions infrastructure.

In [ ]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision_nodes(nodes=["node1"])
await prov.provision()
print(prov.get_ansible_inventory())


In [ ]:
k8s = K8SManagement(preceding=prov)
await k8s.deploy()


Deploy the shared core with the same SMF/UPF metrics image and configuration templates
as the passing RFsim examples. Register all twenty subscribers before starting UEs.

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=k8s)
for component in core.deployment_model.core.components.values():
    component.node_name = "node1"


In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


Start the monolithic gNB and CU first, then the DU once its CU is running.
The local gNB/CU charts include the same metrics-parser sidecars as the existing examples.

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(preceding=core)


In [ ]:
# Select local OAI charts that include the metrics-parser sidecar.
from importlib.resources import files

oai_charts = files("nador") / "assets" / "ran" / "oai"
for name, component in ran.deployment_model.ran.components.items():
    if component.implementation == "oai" and component.type in ("gnb", "cu", "cu_cp", "cu_up", "du"):
        component.custom_chart_path = str(oai_charts / f"oai-{component.type}")
        component.custom_rendered_files = [
            str(oai_charts / "values.yaml.j2"),
            str(oai_charts / "config.yaml.j2"),
        ]


### CSI logging

The monolithic gNB (`node2`) and the DU (`node4`) can; the CU has no PHY, hence no SRS, and log SRS channel estimates per resource block. `enable_csi` selects the `r2labuser/oai-gnb-csi` image
with the same OAI tag for the radios of `CSI_COMPONENTS` only (with an empty list, no CSI and the images without
logger), forces periodic SRS and CSI-RS in the rendered configuration, and mounts `/data/csi` from the node: each
logger writes `/data/csi/csi_per_rb.csv` on its own VM, where the file survives pod restarts. Run this cell before
preparing the radios. RFsim estimates are flat across RBs (no fading model, see RFsim channel below).

`CSI_ENV` passes options to the logger (`CSI_GRANULARITY`, `CSI_SUBCARRIER_SAMPLING`, `CSI_ANTENNA_SELECTION`,
`CSI_PORT_SELECTION`, `CSI_FLUSH_CORE`) and `CSI_COLLECT` sets how each experiment retrieves the CSI; the full option
tables are in `examples/monitoring/multi-gnb-2du/test.ipynb`.

In [ ]:
from nador.csi import CsiCollector, enable_csi

# CSI mode: radios that log (empty list: no CSI) and logger options (empty: defaults, RB granularity)
CSI_COMPONENTS = ['gnb', 'du']
CSI_ENV = {}  # e.g. {"CSI_GRANULARITY": "subcarrier", "CSI_SUBCARRIER_SAMPLING": "2"}
# CSI retrieval by each ExperimentRun: whole file + per-section files, at the lowest priority on the radio VM
CSI_COLLECT = {"fetch_full": True, "window_types": "section", "padding_seconds": 0}

enable_csi(ran, components=CSI_COMPONENTS, env=CSI_ENV)
for name in CSI_COMPONENTS:
    component = ran.deployment_model.ran.components[name]
    print(f"{name} on {component.node_name}: {component.image.repository}:{component.image.image_tag}",
          component.extra_env_vars)

# One collector per CSI component, reused by every ExperimentRun below.
CSI_COLLECTORS = [CsiCollector(ran, name, **CSI_COLLECT) for name in CSI_COMPONENTS]

#### Default CSI settings

With the CSI cells as provided (`CSI_ENV = {}`), the settings below apply. Change them in the cells named in the last column; logger settings take effect when the radios start, so set them before preparing the radios.

| Setting | Value in this notebook | Where it is set |
|---|---|---|
| Granularity (`CSI_GRANULARITY`) | `rb`: one complex value per RB | `CSI_ENV` (logger default) |
| Subcarrier sampling (`CSI_SUBCARRIER_SAMPLING`) | 1 (only used with `subcarrier`) | `CSI_ENV` (logger default) |
| Antennas / ports (`CSI_ANTENNA_SELECTION` / `CSI_PORT_SELECTION`) | all | `CSI_ENV` (logger default) |
| Writer thread CPU (`CSI_FLUSH_CORE`) | 32 (left unpinned with a warning when the CPU does not exist) | `CSI_ENV` (logger default) |
| SRS period | chosen by OAI: smallest allowed period covering TDD period x maximum number of UEs per gNB (measured in the rfsim scenarios: 160 slots, 80 ms) | OAI gNB/DU (`do_SRS: periodic`) |
| CSI file | `/data/csi/csi_per_rb.csv` on the radio node (hostPath) | `enable_csi` (`output_dir`) |
| Whole CSI file copied per run (`fetch_full`) | yes | `CSI_COLLECT` |
| Windows cut (`window_types`, `padding_seconds`) | `section`, 0 s | `CSI_COLLECT` |
| Reading CPUs on the radio node (`cpu_list`) | any (lowest priority) | `CSI_COLLECT` |
| Temporary files on the radio node (`cleanup_remote`) | removed | `CsiCollector` default |
| Clock offset correction | applied only above 0.5 s (offset and drift recorded per run) | `ExperimentRun(csi_clock_threshold_s=0.5)` |
| Live view | on `node1`, ports `8501`, `8502`, 120 s window, 10 s refresh, live copy restarted beyond 1024 MB, visualizer v9.0 (`oai-csi-logging` `csi-v3.1`) | `CSI_LIVE`, `CSI_LIVE_PORTS` |
| RFsim channel (`RFSIM_CHANNEL`) | `2x2+awgn`: 2 gNB/DU receive antennas, single-port SRS (see RFsim channel) | RFsim channel cell |

### RFsim channel

`RFSIM_CHANNEL` in the next cell selects the radio channel of the OAI gNBs/DUs with a simulated RU and of their UEs:

| `RFSIM_CHANNEL` | gNB/DU antennas | UE antennas (transmit / receive) | Channel model |
|---|---|---|---|
| `"2x2+awgn"` (default) | 2 (2 PUSCH ports, 2 MIMO layers) | 2 / 1 | AWGN, noise power -30 dB, no path loss |
| `"2x2+epa"` (still to be debugged) | 2 (2 PUSCH ports, 2 MIMO layers) | 2 / 1 | 3GPP EPA fading (7 paths up to 410 ns, 2x2 correlation) |
| `"awgn"` | 1 | 1 / 1 | the same AWGN model |
| `"ideal"` | 1 | 1 / 1 | none |

`"2x2+awgn"` is the radio configuration of the 5g_ansible rfsim scenario (`oai5g-rru` `gnb-config-rfsim.yaml` /
`du-config-rfsim.yaml`, UEs with `--ue-nb-ant-tx 2`). The OAI UE takes its antennas from the RU of its configuration,
so they are set on the simulated RU of each radio. Set it before preparing the radios and before the UE manager is
created.

What to expect in the CSI: OAI's AWGN channel is the identity matrix plus white noise (`random_channel.c`), and the
UEs send a single-port SRS (OAI UE capabilities without `--uecap_file`). In `"2x2+awgn"`, gNB/DU antenna 0 receives the
SRS and antenna 1 stays near 0 (it is linked to the second UE antenna, which sends no SRS). The estimates are flat
across RBs in every profile: OAI adds `10**(noise_power_dB/10) * 256` of noise per int16 sample, negligible at -30 dB
(0 dB already prevents UE attachment). `"2x2+epa"` uses a fading model instead: both gNB/DU antennas receive the SRS through different,
frequency-selective channels. RFsim draws the channel once per UE connection (OAI `simulator.cpp`), so the estimates
do not evolve over time for a connection, and the multipath convolution costs much more CPU than AWGN: check that the
radio VMs keep up (frame advance in the CSI, UE attachment).

Known issues with the OAI images used here (`develop-f80a883`, 2026-10-16), observed on SLICES BI:

- `"2x2+epa"` is still to be debugged. The EPA models load (`Model rfsimu_channel_ue0 type EPA allocated`), but the
  gNB/DU softmodem crashes as soon as a UE connects, right after `Random channel rfsimu_channel_ue0 in rfsimulator
  activated`: the kernel log of the radio VM shows `ru_thread: segfault at 0 ... error 4 in librfsimulator.so`, always
  at offset `0xf11b` (NULL pointer read in the rfsimulator when it applies the EPA channel). The UEs then lose their
  connection (`write() failed, errno(104)`) and no UE attaches. The configuration rendered by nador is the same
  as `"2x2+awgn"` except the channel type.
- `"2x2+awgn"`: the UEs attach (gNB/DU 2x2, UEs with 2 transmit and 1 receive antennas), but a UE softmodem may crash
  under sustained traffic (iperf): `DL__actor: segfault ... error 6 in libc.so.6`, always at offset `0x17173a`
  (invalid write while processing the downlink). Use `"awgn"` or `"ideal"` for long traffic experiments until it is
  understood.

In [ ]:
from nador.csi import set_rfsim_channel

# RFsim channel: "2x2+awgn" (default, as the 5g_ansible rfsim scenario), "2x2+epa" (fading, still to be debugged), "awgn" (1x1) or "ideal"
RFSIM_CHANNEL = "2x2+awgn"
set_rfsim_channel(ran, RFSIM_CHANNEL)
for name, component in ran.deployment_model.ran.components.items():
    advanced = component.advanced_config
    if advanced is not None and advanced.channelmod:
        antennas = (component.extra_env_vars or {}).get("RFSIM_ANTENNAS", "1")
        print(f"{name}: {antennas}x{antennas}, {advanced.channelmod['modellist_rfsimu_1'][0]['type']} channel")

In [ ]:
await ran.prepare(components=["gnb", "cu"])
await ran.install(components=["gnb", "cu"])
await ran.deploy(components=["du"])


Install monitoring **before UE attachment** so the AMF/SMF sniffers observe registration
and session establishment. Both `gnb` and `cu` are packet targets; DU traffic reaches
N3 through its CU. `eth0` is used with the default pod network.

The OAI calculator provides monolithic MAC KPIs. The CU parser is also scraped,
but does not provide DU MAC statistics.

In [ ]:
from nador import MonitoringManagement
from nador.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")
monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)
await monitoring.deploy()


Optional: run this cell to open Grafana from your laptop with the current inventory connection.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = [
    "ssh",
    "-N",
    "-T",
    "-S",
    "none",
    "-o",
    "ControlMaster=no",
    "-o",
    "ControlPersist=no",
    "-o",
    "BatchMode=yes",
    "-o",
    "ExitOnForwardFailure=yes",
    "-o",
    "ConnectTimeout=10",
]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += [
    "-p",
    str(ssh.port),
    "-L",
    f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
    f"{ssh.user}@{ssh.hostname or control}",
]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))


### Live CSI view

One Streamlit view per CSI radio runs on `node1`, never on a radio VM: it follows the radio VM's CSI file over
SSH, where only `sshd` and `tail -F` run at the lowest priority. The view shows a sliding window (120 s by default) in
local time; start it before attaching UEs so that SRS reports appear as they attach. The first start installs the
viewer (about one minute); the tunnel is opened on this machine. Stop it with the cell before the VMs are destroyed.

In [ ]:
from nador.csi import CsiLiveView
from IPython.display import Markdown, display

CSI_LIVE = {"window_s": 120, "refresh_s": 10, "max_mb": 1024}
CSI_LIVE_PORTS = {'gnb': 8501, 'du': 8502}
lives = {
    name: CsiLiveView(ran, live_node='node1', component=name, port=CSI_LIVE_PORTS[name], **CSI_LIVE)
    for name in CSI_COMPONENTS
}
links = []
for name, view in lives.items():
    url = await view.start()
    links.append(f"[CSI {name}]({url})")
display(Markdown(" · ".join(links)))

Start the twenty UEs sequentially to avoid starting all RFsim clients at once.
Their IMSIs are already registered. The core/RAN manager chain stays separate from
monitoring; monitoring observes the resulting pods and sessions.

In [ ]:
from orchestrator.deployment import UEManagement

ue = UEManagement(preceding=ran)
for client in CLIENTS:
    print(f"Deploying {client}")
    await ue.deploy(components=[client], quiet=True)


Check that each CSI logger runs and that its file grows: the logs show the logger start, and each `csi_per_rb.csv`
must grow between the two listings (the logger flushes every 5 seconds).

In [ ]:
from orchestrator.deployment import ProvisioningManagement

for name in CSI_COMPONENTS:
    logs = await ran.get_logs(components=[name], quiet=True)
    print(f"--- {name}")
    print("\n".join(line for line in logs.splitlines() if "[CSI]" in line or "Init: N_RB_DL" in line))

csi_nodes = {name: ran.deployment_model.ran.components[name].node_name for name in CSI_COMPONENTS}
nodes = ProvisioningManagement(preceding=ran)
for _ in range(2 if CSI_COMPONENTS else 0):  # sizes must grow between the two listings
    listing = await nodes.run_node_command(command=["ls", "-l", "/data/csi/csi_per_rb.csv"],
                                           nodes=sorted(set(csi_nodes.values())), become=True, quiet=True, flatten=False)
    for name, node in csi_nodes.items():
        print(f"{name} ({node}): {listing[node].strip()}")
    await asyncio.sleep(10)

Read the assigned UE tunnel addresses and install iperf3 on the server VM.
Each UE must have a distinct address. This is also where you can inspect the two groups
before choosing traffic for your custom scenario.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = "node1"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5
UE_INTERFACE = "oaitun_ue1"

server_node = ue.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(ue.deployment_model.params.ansible.runner_path) / "iperf-server")


await run_ansible_playbook(
    playbook=[
        {
            "name": "Install iperf3 on the server VM",
            "hosts": SERVER_NODE,
            "gather_facts": False,
            "become": True,
            "tasks": [
                {
                    "name": "Install iperf3",
                    "ansible.builtin.apt": {
                        "name": "iperf3",
                        "state": "present",
                        "update_cache": True,
                        "cache_valid_time": 3600,
                    },
                }
            ],
        }
    ],
    inventory=server_inventory,
    runner_path=server_runner,
    quiet=True,
)
addresses = await ue.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], components=CLIENTS, quiet=True, flatten=False
)

ue_ips = {}
for name in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[name]) for a in device["addr_info"] if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{name}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[name] = ipv4[0]

if len(set(ue_ips.values())) != len(CLIENTS):
    raise ValueError("Select UEs with different tunnel addresses")

if not QUERIES_FILE.is_file():
    raise FileNotFoundError(QUERIES_FILE)

print(f"iperf server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


Traffic helpers use the same UE-to-VM TCP commands as the passing RFsim examples.
Uplink sends from the UE; downlink adds iperf's reverse mode.

In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await ue.run_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        components=[client],
        quiet=True,
        flatten=False,
    )
    return json.loads(output[client])


Define the measured workload here. By default, every UE runs 20 seconds uplink and
20 seconds downlink, sequentially: **40 steps, 13 minutes 20 seconds of requested traffic**,
plus command and collection overhead. For a short smoke run, set
`TRAFFIC_CLIENTS = [UE_GROUPS["monolithic"][0], UE_GROUPS["split"][0]]`.
All twenty UEs remain deployed regardless of this traffic selection.

In [ ]:
TRAFFIC_CLIENTS = CLIENTS
SCENARIO = [
    {"name": f"{client}_{direction}", "client": client, "direction": direction, "duration": DURATION}
    for client in TRAFFIC_CLIENTS
    for direction in ("uplink", "downlink")
]


Run the workload and collect artifacts. `ExperimentRun` saves the timeline, query exports
and iperf reports. The group and address files identify which gNB each client used.

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="multi-gnb",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
    csi=CSI_COLLECTORS,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)
    experiment.save_json("ue_groups.json", UE_GROUPS)
    experiment.save_json("ue_addresses.json", ue_ips)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(test["name"]):
            report = await iperf_client(
                client=test["client"],
                direction=test["direction"],
                duration=test["duration"],
                port=port,
            )

        experiment.save_json(f"{test['name']}.json", report)
        if report.get("error"):
            raise RuntimeError(f"{test['name']}: {report['error']}")

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")


Check the CSI collected by the run: per radio, its node clock (the offset to this machine is only applied above
0.5 s), the rows and 5-second batches read, and the per-section files under `csi/<component>/by_window/section/`.

In [ ]:
meta = json.loads((output_directory / "experiment_metadata.json").read_text())
print(meta["status"], meta["collection_errors"])
for name, m in meta.get("csi", {}).items():  # empty without CSI
    print(f"--- {name} on {m['node']}")
    print("  clock:", {k: m.get(k) for k in ("clock_offset_applied", "clock_offset_applied_s", "clock_drift_s", "warnings")})
    print("  start:", {k: round(m["clock_start"][k], 3) for k in ("offset_s", "rtt_s", "uncertainty_s")})
    print("  data: ", {k: m.get("summary", {}).get(k) for k in ("rows", "markers", "dropped_rows", "windows_written", "windows_empty")})
print(sorted(str(p.relative_to(output_directory)) for p in (output_directory / "csi").rglob("*.gz")))

In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")


### End of the experiment

**Run All** stops at the next cell, so that the end-of-experiment cells below (stop of the live CSI views, UE
disconnection, release of the nodes) only run when you run them yourself, one at a time. The kernel keeps its state:
run the cells below with Shift+Enter when the experiment is over.

In [ ]:
class StopRunAll(Exception):
    """Ends "Run All" here; IPython shows only the message below (no traceback)."""

    def _render_traceback_(self):
        return [str(self)]


raise StopRunAll("Run All stops here. When the experiment is over, run the cells below one at a time.")

Stop the live CSI views before destroying the VMs: this closes the tunnels and revokes their SSH keys on the radio VMs.

In [ ]:
for view in globals().get("lives", {}).values():
    await view.stop()

When finished, run the next cell to destroy this example's VMs and their workloads.
If you want to keep 5G running and remove only monitoring, run
`await monitoring.remove()` instead and leave the VM-cleanup cell unexecuted.

In [ ]:
await prov.unprovision()
